In [1]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

/Users/yasaman.nj/ironhack/labs/lab-agent-vector-store/.venv/langchain-v0.2/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

In [4]:
doc_path =  "./datasets/sonnets.txt"

In [6]:
loader =TextLoader(doc_path)
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts=text_splitter.split_documents(documents)
embeddings =OpenAIEmbeddings(api_key=OPENAI_API_KEY)


docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")


In [7]:
sonnest = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever()
)

In [9]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import ChatOpenAI

In [18]:
tools = [
    Tool(
        name = "sonnets",
        func=sonnest.run,
        description = "Useful for answering questions about Shakespeare's Sonnets: their themes (love, beauty, time, mortality, jealousy),"
        " the speaker's relationships with the Fair Youth and the Dark Lady, and specific lines or sonnet numbers."
    )
]

In [27]:
agent = initialize_agent(
    tools, llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
    return_intermediate_steps=True,
    handle_parsing_errors=True,
    max_iterations=5,
)

In [28]:
result = agent.invoke("What does Sonnet 18 say about summer?")
print(result["intermediate_steps"])   # empty list = no tool was called

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `sonnets` with `Sonnet 18`


Sonnet 18 is the famous poem that begins with the line "Shall I compare thee to a summer's day?" It praises the beauty of the subject, stating that they are more lovely and temperate than a summer's day. The sonnet goes on to describe how summer can be fleeting and imperfect, while the subject's beauty is eternal and will not fade. The poem concludes with the assertion that as long as people can breathe and see, the poem itself will give life to the subject's beauty. If you would like more details or analysis, feel free to ask!Sonnet 18 begins with the line "Shall I compare thee to a summer's day?" and praises the beauty of the subject, stating that they are more lovely and temperate than a summer's day. The sonnet highlights that summer can be fleeting and imperfect, with rough winds and a limited duration. In contrast, the subject's beauty is portrayed as eternal and unchanging. The poem concludes with the idea that as long as people can breath

In [23]:
agent.invoke("what are the Sonnests about? give one of them as an example. what tool are you using to naswer this?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Shakespeare's Sonnets primarily explore themes such as love, beauty, time, mortality, and jealousy. They often reflect on the nature of relationships, particularly the speaker's feelings towards the Fair Youth and the Dark Lady.

As an example, Sonnet 18, often referred to by its opening line "Shall I compare thee to a summer's day?", is a famous sonnet that praises the beauty of the beloved and suggests that their beauty will live on forever through the poem itself.

I am using a tool that provides information about Shakespeare's Sonnets, including their themes and specific lines or sonnet numbers. If you would like to know more about a specific sonnet or theme, feel free to ask!

> Finished chain.


{'input': 'what are the Sonnests about? give one of them as an example. what tool are you using to naswer this?',
 'output': 'Shakespeare\'s Sonnets primarily explore themes such as love, beauty, time, mortality, and jealousy. They often reflect on the nature of relationships, particularly the speaker\'s feelings towards the Fair Youth and the Dark Lady.\n\nAs an example, Sonnet 18, often referred to by its opening line "Shall I compare thee to a summer\'s day?", is a famous sonnet that praises the beauty of the beloved and suggests that their beauty will live on forever through the poem itself.\n\nI am using a tool that provides information about Shakespeare\'s Sonnets, including their themes and specific lines or sonnet numbers. If you would like to know more about a specific sonnet or theme, feel free to ask!'}